# Schwarzman Scholars Open Data — Full Control Overview
**1,497 scholars · 74 public intros · 285 bios · 4 archetypes · Warmth v2 + Whisper**

This notebook oversees *everything* we simulated — no black box. Run top-to-bottom for full control. All paths are relative to the repo root (`..` from `notebooks/`).

> **Run:** `pip install -r ../requirements.txt` (or `pandas matplotlib scikit-learn nltk`) then `jupyter lab notebooks/schwarzman_overview.ipynb`

Shorthand legend: `data/video_legend.csv` → `AA18` = Abdullah Almiqasbi 2018 (First+Last+cohort). Use it as the axis label on every PNG so 74 names stay readable.


In [ ]:
# --- 0) Setup — robust ROOT finder (works no matter where Jupyter was launched) ---
import pathlib, sys
def _find_root():
    # Try common locations first (absolute fallback to your physical repo)
    candidates = [
        pathlib.Path.cwd(),
        pathlib.Path.cwd() / "SchwarzmanScholarsOpenData",
        pathlib.Path.home() / "Desktop" / "sop" / "SchwarzmanScholarsOpenData",
        pathlib.Path("..").resolve(),
        pathlib.Path("../..").resolve(),
        pathlib.Path("../../..").resolve(),
        pathlib.Path(".").resolve(),
    ]
    # walk up from each candidate
    for cand in candidates:
        p = cand
        for _ in range(7):
            if (p / "data" / "schwarzman_scholars_dataset.csv").exists():
                return p
            p = p.parent
    # last resort: brute find from home/Desktop
    for p in (pathlib.Path.home() / "Desktop" / "sop").rglob("schwarzman_scholars_dataset.csv"):
        return p.parent.parent
    return pathlib.Path.home() / "Desktop" / "sop" / "SchwarzmanScholarsOpenData"

ROOT = _find_root()
DATA = ROOT / "data"
AD = ROOT / "analytics_dashboard"
print(f"ROOT → {ROOT}")
print(f"DATA exists? {(DATA / 'schwarzman_scholars_dataset.csv').exists()} — {(DATA / 'schwarzman_scholars_dataset.csv').stat().st_size if (DATA / 'schwarzman_scholars_dataset.csv').exists() else 'missing'} bytes")
print(f"AD exists? {AD.exists()} — {len(list(AD.glob('*.png'))) if AD.exists() else 0} PNGs")
import pandas as pd, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


## 0) One Theme for the whole notebook

Run the next cell once. It sets the Theme for every figure after it — bars, PCA, warmth scatters, treemaps. The notebook then reads as one publication, not 12 mismatched charts.

| Ref | What it is | How Grammar reads it | What we do |
|---|---|---|---|
| **Hybrid treemap** (`schwarzman_hybrid_4K.png`) | White tiles, thin `#E2E8F0` strokes, area = frequency, black sans labels, no hue | Data = `bios_words.csv` · Aes = area + position · Geom = rect · Theme = minimal, white | Encode with length/area, not color. One ramp only: `slate #0F172A → #CBD5E1`, darkest = most. No Blues, viridis, crest. |
| **L-system cover** (`l-system-schwarzman.png`) | Open white page, thin black branching, crimson serif title, tiny cyan dot | Visual Hierarchy: use color only for the finding (title, mean line, highlight) | Display title = serif `#A51C30`; axis/body = sans `#334155`/`#64748B`; grid = y-only dashed `#E2E8F0` at 0.35; no top/right spines. |

**Three rules we follow:** 1) Grammar of Graphics — we fix Theme once (`schwarzman_style.ENSEMBLE_RCPARAMS`), vary only Data/Aes/Geom. 2) Visual Hierarchy — color highlights the finding, not every category (`PALETTE_CAT4` = three slates + one crimson). 3) Small multiples — facet rather than overload one plot (see archetype panel in the token sheet).

| Token | Value | Where it goes |
|-------|-------------------------------|-------------|
| `CANVAS` | `#FFFFFF` | `figure.facecolor`, `savefig.facecolor` — pure white, both refs |
| `PANEL` | `#F8FAFC` | optional `axes.facecolor` for bar/line panels (treemap stays `#FFFFFF`) |
| `BORDER` | `#E2E8F0` | spines, grid, treemap rect strokes, outer page border |
| `INK` | `#0F172A` | titles (default), bar fills (darkest), tree ink |
| `SLATE / MUTED / FAINT` | `#334155` / `#64748B` / `#94A3B8` | axis labels / ticks / footnotes |
| `SCHWARZMAN_RED` | `#A51C30` | display title, highlight bar/dot, mean line — use sparingly |
| `PALETTE_MONO_SEQ` | `#0F172A → #CBD5E1` | every sequential bar/line/heatmap (ordered darkest→lightest) |
| `PALETTE_CAT4` | `[#0F172A, #334155, #64748B, #A51C30]` | 4 archetypes — 3 slates + 1 crimson |
| `CMAP_WARMTH` | `F1F5F9 → #A51C30` | warmth scatter `c=warmth` — replaces `Blues/viridis` |
| `SANS / SERIF` | `Helvetica Neue, Inter` / `Garamond, Georgia` | body vs. display — Ref 2 hierarchy |
| `FIGSIZE_WIDE` | `(11,5)` | default for cohort bars, warmth scatters — generous whitespace like Ref 2 |

> Learn it once: `fig, ax = plt.subplots(figsize=ss.FIGSIZE_WIDE)` → `ax.bar / ax.scatter / ...` → `ss.style_axes(ax, title=..., subtitle=..., source=...)` → `ss.save(fig, AD / "my_plot.png")`. Keep the token sheet below visible while you plot.


In [ ]:
# ── 0) ENSEMBLE PREAMBLE — run this cell once, everything after inherits Ref1×Ref2 ──
# Unites the hybrid treemap (structure) + L-system cover (narrative) into one Theme.
# Frameworks: Grammar of Graphics · Visual Hierarchy · Small Multiples (Faceting)
# Tokens live in schwarzman_style.py — single source of truth for notebook + generate_plots.py.
import sys
from pathlib import Path
# ensure repo root is on sys.path whether notebook launched from notebooks/ or repo root
ROOT_HINT = Path.cwd()
for p in [ROOT_HINT, ROOT_HINT / "SchwarzmanScholarsOpenData", Path("..").resolve(), Path("../..").resolve(), Path.home() / "Desktop" / "sop" / "SchwarzmanScholarsOpenData"]:
    if (p / "schwarzman_style.py").exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        break
# also add ROOT itself if ROOT already defined from cell above
try:
    if 'ROOT' in globals() and (Path(ROOT) / "schwarzman_style.py").exists() and str(Path(ROOT)) not in sys.path:
        sys.path.insert(0, str(Path(ROOT)))
except: pass

import schwarzman_style as ss
import matplotlib.pyplot as plt
import matplotlib as mpl
from IPython.display import HTML, display, Image

# 1 · Global Theme — Grammar of Graphics "Theme" step, applied once
ss.apply_style()
print(f"Ensemble applied — figure {mpl.rcParams['figure.facecolor']} · axes {mpl.rcParams['axes.facecolor']} · red {ss.SCHWARZMAN_RED}")
print(f"Palettes — mono {ss.PALETTE_MONO_SEQ[:3]}... · cat4 {ss.PALETTE_CAT4} · cmap {ss.CMAP_WARMTH.name}")

# 2 · Notebook CSS — makes markdown headings follow the same typographic hierarchy as the plots
#    (Ref 2 editorial: h1 serif crimson, h2 slate, body sans slate, code faint)
display(HTML(f"""
<style>
  /* keep Jupyter's layout but recolor the editorial hierarchy */
  .jp-Notebook h1, .text_cell_render h1 {{
    font-family: Garamond, Georgia, 'Times New Roman', serif !important;
    color: {ss.SCHWARZMAN_RED} !important;
    font-weight: 700 !important; letter-spacing: -0.01em; border-bottom: none !important;
  }}
  .jp-Notebook h1::after {{
    content: ""; display: block; width: 42px; height: 2px; margin-top: 6px;
    background: {ss.SCHWARZMAN_RED}; border-radius: 1px;
  }}
  .jp-Notebook h2, .text_cell_render h2, .jp-Notebook h3, .text_cell_render h3 {{
    font-family: 'Helvetica Neue', Inter, 'DejaVu Sans', Arial, sans-serif !important;
    color: {ss.INK} !important; font-weight: 700 !important;
  }}
  .jp-Notebook h2, .text_cell_render h2 {{ font-size: 1.18em !important; }}
  .text_cell_render {{ color: {ss.SLATE}; line-height: 1.55; }}
  .text_cell_render a {{ color: {ss.SCHWARZMAN_RED}; text-decoration: none; border-bottom: 1px solid {ss.BORDER}; }}
  .text_cell_render blockquote {{ border-left: 3px solid {ss.BORDER}; color: {ss.MUTED}; background: {ss.PANEL}; padding: 8px 14px; }}
  .text_cell_render code {{ color: {ss.INK}; background: {ss.PANEL}; border: 1px solid {ss.BORDER}; }}
  /* narrower, editorial line-length like Ref 2 — generous whitespace */
  #notebook-container, .jp-Notebook {{ background: {ss.CANVAS} !important; }}
  .jp-Cell {{ border: 1px solid transparent; }}
</style>
"""))

# 3 · Visual cheat-sheet — keep this visible while you plot (tokens, palettes, type)
#    Saved to analytics_dashboard/style_tokens.png so generate_plots.py and README can reuse it.
try:
    TOKENS_PNG = Path(ROOT) / "analytics_dashboard" / "style_tokens.png" if 'ROOT' in globals() else Path("../analytics_dashboard/style_tokens.png")
    TOKENS_PNG.parent.mkdir(parents=True, exist_ok=True)
    fig = ss.preview_tokens(save_path=str(TOKENS_PNG))
    # preview_tokens saved directly when given a path; otherwise it returns fig
    if fig is not None and not isinstance(fig, str):
        # already saved inside; display from file
        display(Image(filename=str(TOKENS_PNG), width=900))
    else:
        display(Image(filename=str(TOKENS_PNG), width=900))
    print(f"Token sheet → {TOKENS_PNG}")
except Exception as e:
    # fallback: just show the figure in-memory
    try:
        fig = ss.preview_tokens()
        display(fig)
    except Exception as e2:
        print("token sheet fallback:", e, e2)

# 4 · One-liner sanity check — every later cell should start with this pattern:
#    fig, ax = plt.subplots(figsize=ss.FIGSIZE_WIDE)
#    ss.bar(ax, labels, values)  or  ax.scatter(... , cmap=ss.CMAP_WARMTH)
#    ss.style_axes(ax, title="...", subtitle="...", source="data/...")
#    ss.save(fig, AD / "my_plot.png")
print("Preamble ready — next cells inherit: plt.rcParams + seaborn theme + notebook CSS. Build plots with ss.style_axes() + ss.save().")


## 1) At a glance — who is in the data

1,497 rows. 74 videos (4.9%). 285 bios (19%). This is the observable slice, not the whole pool.


In [ ]:
df = pd.read_csv(DATA/"schwarzman_scholars_dataset.csv", encoding="utf-8")
print(f"rows {len(df)} | videos {df['has_intro_video'].sum()} | bios {df['bio'].notna().sum()}")
df["has_intro_video"] = pd.to_numeric(df["has_intro_video"], errors="coerce").fillna(0).astype(int)
# top countries / unis — same as README
print(df["country"].value_counts().head(8).to_string())
print(df["university"].value_counts().head(8).to_string())
# show the hybrid treemap (the image you shared)
from IPython.display import Image, display
try:
    display(Image(filename=str(AD/"schwarzman_hybrid_4K.png"), width=800))
except: 
    print("Hybrid treemap at analytics_dashboard/schwarzman_hybrid_4K.png — open it for USA/China/global/policy blocks")


### 1a) Baselines: who feeds the program and who is visible on video

1,497 rows in `schwarzman_scholars_dataset.csv`. Bars show count (length), donuts show share (angle). Color stays ink and border gray; crimson only marks the 4.9% video slice. Tables list the numbers so the page stays useful on GitHub without hovering.

Three baselines matter:
- **Universities** — concentrated at the top (Harvard 86) but long tail matters: top 15 is only 25% of 1,497.
- **Countries** — US 617 + China 300 = 61%. After that every country is under 4%.
- **Video visibility** — 74 public intros out of 1,497 (4.9%). Warmth results come from this small, Western-skewed slice. Report the denominator.


In [ ]:
# Descriptive data — feeder unis, feeder countries, video public % (ensemble, no rainbow)
import pandas as pd, matplotlib.pyplot as plt, pathlib
from IPython.display import HTML, display, Image
# ensure ensemble already applied (cell 03), otherwise re-apply lightly
try:
    import schwarzman_style as _ss
    _ss.apply_style()
except: 
    import schwarzman_style as _ss
    pass

# 1 — Top feeder universities (top 15) + table
top_unis = df["university"].dropna()
top_unis = top_unis[top_unis.str.strip().str.len()>2].value_counts().head(15)
# HTML table with %
tbl_unis = pd.DataFrame({"university": top_unis.index, "scholars": top_unis.values})
tbl_unis["share"] = (tbl_unis["scholars"]/len(df)*100).round(1)
tbl_unis["cumulative"] = tbl_unis["share"].cumsum().round(1)
display(HTML('<h4 style="color:#0F172A; font-family: Helvetica Neue, sans-serif; margin:12px 0 4px;">Top 15 feeder universities — 1,497 scholars</h4>'))
display(HTML(
    '<div style="max-height:320px; overflow:auto; border:1px solid #E2E8F0; border-radius:8px;">'
    + tbl_unis.to_html(index=False, classes="ensemble-table", float_format="%.1f")
      .replace('<table ', '<table style="font-size:11px; border-collapse:collapse; width:100%;" ')
      .replace('<th>', '<th style="background:#0F172A; color:white; padding:6px 8px; text-align:left; position:sticky; top:0;">')
      .replace('<td>', '<td style="padding:5px 8px; border-bottom:1px solid #F1F5F9; color:#334155;">')
    + '</div>'
    + '<div style="font-size:10px; color:#64748B; font-style:italic; margin-top:4px;">Harvard 86 (5.7%) leads, but top 15 = 379/1497 = 25.3% — the other 74% is dispersed (Fresno CS27, Montana DM27 win via Tech/Climate).</div>'
))
fig, ax = plt.subplots(figsize=_ss.FIGSIZE_TALL)
_ss.bar(ax, top_unis.index[::-1], top_unis.values[::-1], orient="h")  # reversed so largest on top after invert
for i, v in enumerate(top_unis.values[::-1]):
    ax.text(v+0.8, i, str(v), va="center", fontsize=7, color=_ss.SLATE)
_ss.style_axes(ax, title="Top 15 Feeder Universities (raw string)", subtitle="1497 scholars · bar length = count, hue stays ink — dispersion is the finding", source="data/schwarzman_scholars_dataset.csv")
ax.set_xlabel("Scholars (n)")
plt.tight_layout(); plt.savefig(AD/"top_unis.png", dpi=_ss.SAVEDPI, bbox_inches="tight", facecolor=_ss.CANVAS); plt.close()
display(Image(filename=str(AD/"top_unis.png"), width=800))

# 2 — Top feeder countries (top 10) + table + bridge share
top_countries = df["country"].dropna()
top_countries = top_countries[~top_countries.str.strip().isin(["0","and"])].value_counts().head(10)
tbl_cty = pd.DataFrame({"country": top_countries.index, "scholars": top_countries.values})
tbl_cty["share"] = (tbl_cty["scholars"]/len(df)*100).round(1)
display(HTML('<h4 style="color:#0F172A; font-family: Helvetica Neue, sans-serif; margin:18px 0 4px;">Top 10 feeder countries — bridge test</h4>'))
display(HTML(
    '<div style="max-height:300px; overflow:auto; border:1px solid #E2E8F0; border-radius:8px;">'
    + tbl_cty.to_html(index=False, classes="ensemble-table", float_format="%.1f")
      .replace('<table ', '<table style="font-size:11px; border-collapse:collapse; width:100%;" ')
      .replace('<th>', '<th style="background:#0F172A; color:white; padding:6px 8px; text-align:left;">')
      .replace('<td>', '<td style="padding:5px 8px; border-bottom:1px solid #F1F5F9; color:#334155;">')
    + '</div>'
    + '<div style="font-size:10px; color:#64748B; font-style:italic; margin-top:4px;">US 617 (41.2%) + China 300 (20.0%) = 61.2% — the bridge is the program. UK 50 (3.3%) next. 74 countries total.</div>'
))
fig, ax = plt.subplots(figsize=(11,6))
_ss.bar(ax, tbl_cty["country"][::-1], tbl_cty["scholars"][::-1], orient="h")
for i, v in enumerate(tbl_cty["scholars"][::-1]):
    ax.text(v+3, i, f"{v} ({v/len(df)*100:.1f}%)", va="center", fontsize=7, weight="bold", color=_ss.INK)
_ss.style_axes(ax, title="Top 10 Countries of Origin — US-China bridge 61%", subtitle="617 US + 300 CN of 1497 · monochrome, length is the encoding", source="data/schwarzman_scholars_dataset.csv (74 countries)")
ax.set_xlabel("Scholars")
plt.tight_layout(); plt.savefig(AD/"top_countries.png", dpi=_ss.SAVEDPI, bbox_inches="tight", facecolor=_ss.CANVAS); plt.close()
display(Image(filename=str(AD/"top_countries.png"), width=800))

# 3 — Video public share (the bias you must disclose)
n_total=len(df); n_vid=int(df["has_intro_video"].sum()); n_novid=n_total-n_vid; pct=n_vid/n_total*100
display(HTML(f'<h4 style="color:#0F172A; font-family: Helvetica Neue, sans-serif; margin:18px 0 4px;">Public videos — coverage & bias</h4><div style="background:#F8FAFC; border:1px solid #E2E8F0; border-radius:8px; padding:10px 14px; font-size:12px; color:#334155;"><b style="color:#A51C30; font-size:15px;">{n_vid} / {n_total} = {pct:.1f}%</b> have a public 1-min intro (YT). The other {n_novid} (95.1%) are dark — any “warmth” finding is <b>sharers-only</b> (Western-skewed: US 44.6% of videos vs 41.2% of scholars, CN 2.7% vs 20.0%). Always report the denominator.</div>'))
fig, ax = plt.subplots(figsize=(6.5,4.8))
colors=[_ss.INK, _ss.BORDER]
wedges, texts, autotexts = ax.pie([n_vid, n_novid], labels=[f"Has Video\n{n_vid}", f"No Video\n{n_novid}"], autopct="%1.1f%%", startangle=90, colors=colors, wedgeprops=dict(width=0.44, edgecolor=_ss.CANVAS, linewidth=1.5), textprops=dict(fontsize=9, color=_ss.INK), pctdistance=0.85)
ax.set_title("Intro Video Coverage — 74 / 1497 public", fontsize=11, fontweight="bold", color=_ss.INK, fontfamily=_ss.SANS[0], pad=12)
plt.tight_layout(); plt.savefig(AD/"video_submissions.png", dpi=_ss.SAVEDPI, bbox_inches="tight", facecolor=_ss.CANVAS); plt.close()
display(Image(filename=str(AD/"video_submissions.png"), width=420))
# bonus: videos per cohort bar (shows where YT lives)
fig, ax = plt.subplots(figsize=_ss.FIGSIZE_WIDE)
videos_per_cohort = df[df["has_intro_video"]==1]["cohort_year"].value_counts().sort_index()
all_cohorts = sorted(df["cohort_year"].unique())
videos_per_cohort = videos_per_cohort.reindex(all_cohorts, fill_value=0)
ax.bar(videos_per_cohort.index, videos_per_cohort.values, color=_ss.INK, edgecolor=_ss.CANVAS, linewidth=0.9, zorder=3)
for i, v in enumerate(videos_per_cohort.values):
    if v>0: ax.text(i, v+0.25, str(int(v)), ha="center", fontsize=7, weight="bold", color=_ss.INK)
_ss.style_axes(ax, title="Public Intro Videos per Cohort (n=74)", subtitle="Spread thin — 2023 peaks at 14, 2019 only 3 — scarcity is the structure", source="data/schwarzman_scholars_dataset.csv")
ax.set_ylabel("Videos"); ax.set_xlabel("Cohort Year")
plt.tight_layout(); plt.savefig(AD/"videos_per_cohort.png", dpi=_ss.SAVEDPI, bbox_inches="tight", facecolor=_ss.CANVAS); plt.close()
display(Image(filename=str(AD/"videos_per_cohort.png"), width=800))
print(f"Descriptive v2 done — top_unis 15, top_countries 10, video coverage {pct:.1f}% (n={n_vid}) → analytics_dashboard/*.png refreshed")


## 2) Bios language — 4 archetypes (public sklearn TF-IDF + KMeans)
We cluster 285 bios into 4 doors. No private model — just `pandas + scikit-learn + nltk`.
- TF-IDF 400 words (stopwords + `the/and` stripped, `and 1412 > the 1129` would otherwise dominate)
- KMeans k=4, PCA 2D for `bios_clusters_pca.png`
- Shorthand `AA18` etc. for dots (First+Last, readable at 300 dpi)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.cluster import KMeans

from sklearn.decomposition import PCA



bios = df[df["bio"].notna()].copy()

STOP = ["a","an","the","and","or","but","if","while","of","at","by","for","with","about","into","through","during","before","after","above","below","to","from","up","down","in","out","on","off","over","under","again","further","then","once","here","there","when","where","why","how","all","any","both","each","few","more","most","other","some","such","no","nor","not","only","own","same","so","than","too","very","s","t","can","will","just","don","should","now","is","are","was","were","be","been","being","has","have","had","do","does","did"]

try:

    from nltk.corpus import stopwords

    import nltk

    try: nltk.data.find("corpora/stopwords")

    except: nltk.download("stopwords", quiet=True)

    STOP = list(set(STOP) | set(stopwords.words("english")))

except: pass



vec = TfidfVectorizer(stop_words=STOP, max_features=400, token_pattern=r"[a-z']+", lowercase=True)

X = vec.fit_transform(bios["bio"].astype(str))

kmeans = KMeans(n_clusters=4, random_state=7, n_init=20)

labels = kmeans.fit_predict(X)

bios["cluster"] = labels

bios["shorthand"] = bios["name"].apply(lambda n: (n.split()[0][0]+n.split()[-1][0]).upper() if len(n.split())>1 else n[:2].upper())



terms = vec.get_feature_names_out()

order = kmeans.cluster_centers_.argsort()[:, ::-1]

names = {0:"Health Systems (n27)", 1:"Climate & China Bridge (n70)", 2:"Policy/International (n84)", 3:"Tech-Business Builders (n104)"}

for ki in range(4):

    top = [terms[i] for i in order[ki, :10]]

    print(f"Cluster {ki}: {names[ki]} — top {', '.join(top)} — n={(labels==ki).sum()}")



# PCA plot (saved as analytics_dashboard/bios_clusters_pca.png)

pca = PCA(n_components=2, random_state=7)

X2 = pca.fit_transform(X.toarray())

bios["pca0"], bios["pca1"] = X2[:,0], X2[:,1]



plt.figure(figsize=(12,7))

# Ensemble palette — 3 slates + 1 crimson (Visual Hierarchy: color sparingly)

try:

    colors = ss.PALETTE_CAT4  # from preamble: [ink, slate, muted, crimson]

except:

    colors = ["#0F172A","#334155","#64748B","#A51C30"]

for ki in range(4):

    m = bios["cluster"]==ki

    plt.scatter(bios.loc[m,"pca0"], bios.loc[m,"pca1"], c=colors[ki], label=f"{ki}: {names[ki]}", s=36, alpha=0.78, edgecolor="white", linewidth=0.5)

for _, r in bios.iterrows():

    sh = str(r["shorthand"]).strip()

    if not sh or sh.lower()=="nan":

        continue

    plt.text(r["pca0"], r["pca1"], sh, fontsize=4.5, ha="center", va="center", weight="bold", color=colors[int(r["cluster"])], bbox=dict(facecolor="white", edgecolor="none", alpha=0.58, boxstyle="round,pad=0.08"))



fig = plt.gcf()

ax = plt.gca()

ss.style_axes(ax, grid_axis="both")

ax.set_xlabel("PCA 1"); ax.set_ylabel("PCA 2")

leg = ax.legend(frameon=True, facecolor=ss.CANVAS, edgecolor=ss.BORDER, fontsize=8, loc="upper right")

fig.text(0.07, 0.96, "4 Archetypes — PCA of 285 Bios (TF-IDF 400 + KMeans)", fontsize=12, color=ss.INK, fontweight="bold", fontfamily=ss.SANS[0], ha="left", va="bottom")

fig.text(0.07, 0.92, "All 285 shorthands at 4.5pt · crimson = Tech (largest), ink = Health · white halo keeps dense centre readable", fontsize=8, color=ss.MUTED, fontfamily=ss.SANS[0], ha="left", va="bottom")

fig.add_artist(plt.Line2D([0.07, 0.11], [0.91, 0.91], transform=fig.transFigure, color=ss.SCHWARZMAN_RED, linewidth=1.4))

fig.add_artist(plt.Line2D([0.115, 0.18], [0.91, 0.91], transform=fig.transFigure, color=ss.BORDER, linewidth=0.8))

fig.tight_layout(rect=[0,0,1,0.88])

fig.savefig(AD/"bios_clusters_pca.png", dpi=320, bbox_inches="tight", facecolor=ss.CANVAS); plt.close(fig)

print("saved bios_clusters_pca.png")



# also show pre-saved cluster sizes (now ensemble)

try:

    display(Image(filename=str(AD/"bios_cluster_sizes.png"), width=700))

    display(Image(filename=str(AD/"bios_clusters_by_region.png"), width=700))

except: pass



**Name distributions grouped by cluster** — each is a door you can copy. Full lists in `data/cluster_names/cluster_*.csv`.
- Tech-Business `n104` 36%: `AZ Adele Zhong (CN), AD Akorfa Dagadu (Ghana/MIT)…` — US 38%·CN 15% dispersed
- Policy/Intl `n84` 29%: `AH Aili Hou (US/Columbia), AG Aleena Gul (US/Yale)…` — US 64% Harvard-heavy
- Climate-Bridge `n70` 25%: `AS Ajay Sawant (India), LO Lozangtashi (CN/Tibet)…` — CN 44%
- Health `n27` 9%: `AB Anita Bassey (US), DC Daphne Chebesi (Cameroon)…` — most dispersed, Africa 15%


In [ ]:
# Grouped name tables — read before you write your own bio (ensemble, no truncation)

import pathlib, pandas as pd

from IPython.display import HTML, display

# widen pandas so university names don't collapse to "..."

pd.set_option('display.max_columns', None)

pd.set_option('display.max_colwidth', 60)

pd.set_option('display.width', 1200)



clusters = pd.read_csv(DATA/"bios_clusters.csv", encoding="utf-8")

for ki in sorted(clusters["cluster"].unique()):

    sub = clusters[clusters["cluster"]==ki].sort_values("name")

    print(f"\n--- Cluster {ki}: {names[ki]} ---")

    # region mix for context (now readable)

    region_counts = sub.merge(pd.read_csv(DATA/"bios_clusters_with_region.csv", encoding="utf-8")[["name","region"]], on="name").groupby("region").size()

    print(region_counts.to_string())

    sh_list = [s for s in sub["shorthand"].tolist() if isinstance(s, str) and s.strip()]

    print("shorthands (" + str(len(sh_list)) + " with code):", ", ".join(sh_list[:14]) + (" ..." if len(sh_list)>14 else ""))

    # styled HTML table — head 12, scrollable, ensemble border, so no "header only" illusion

    tbl = sub[["shorthand","name","country","university","cohort_year"]].head(12).fillna("—")

    html = tbl.to_html(index=False, escape=False, classes="ensemble-table")

    # ensemble CSS: header ink #0F172A, border #E2E8F0, zebra faint

    styled = (

        '<div style="max-height:340px; overflow:auto; border:1px solid #E2E8F0; border-radius:8px; margin:8px 0;">'

        + html.replace('<table ', '<table style="font-size:11px; border-collapse:collapse; width:100%; font-family: Helvetica Neue, Inter, sans-serif;" ')

        .replace('<th>', '<th style="background:#0F172A; color:white; padding:6px 8px; text-align:left; position:sticky; top:0;">')

        .replace('<td>', '<td style="padding:5px 8px; border-bottom:1px solid #F1F5F9; color:#334155;">')

        + '</div>'

        + f'<div style="font-size:10px; color:#64748B; font-style:italic;">full {len(sub)} names → data/cluster_names/cluster_{ki}_*.csv — filter by <code>cluster=={ki}</code> to copy sentence shape, not content.</div>'

    )

    display(HTML(styled))

print("\nTip: open any cluster_*.csv for the complete list — this preview shows 12 per door to keep the page readable.")



## 3) All 74 videos — shorthand + where Schwarzman plays vs itself / vs others
`data/video_legend.csv` gives `AA18..XR27` for every intro (73 unique vids, `SL26`/`SL27` share one). Use `shorthand` on every axis so plots stay readable.
- Warmth v2 = `happy*0.9 + neutral*0.25 - fear*0.15 - sad*0.10 +35` (7 frames, fallback Retina→OpenCV→MTCNN) — mean 63.1 (n=24 scored, 50 pending) vs old +30 hack 71.2
- Sentiment = TextBlob on Whisper tiny (>0.10 optimistic) — videos 0.14 vs bios 0.064 flat


In [ ]:
legend = pd.read_csv(DATA/"video_legend.csv", encoding="utf-8")
feat = pd.read_csv(DATA/"video_features_all.csv", encoding="utf-8")
print(f"legend {len(legend)} rows — {legend['warmth_v2'].notna().sum()} scored, {legend['warmth_v2'].isna().sum()} pending")
print(legend.head(5).to_string(index=False))
# region share vs all 1497
all_regions = df["country"].map(lambda c: "US" if c=="United States of America" else "CN" if c=="China" else "Other").value_counts(normalize=True).round(3)
print("All 1497 US/CN/Other share:", all_regions.to_dict())
print("YT 74 region share:", legend["region"].value_counts(normalize=True).round(3).to_dict())
# cohort share
print(legend["cohort"].value_counts().sort_index().to_string())
# show all74 shorthand plots we pre-generated
try:
    display(Image(filename=str(AD/"warmth_vs_sentiment_all74_shorthand.png"), width=900))
    display(Image(filename=str(AD/"charisma_by_cohort_all74_shorthand.png"), width=900))
except: print("plots at analytics_dashboard/*all74_shorthand.png")


### How Schwarzman plays
- **vs itself (video sharers vs all 1497):** US 44.6% vs 41.2% even, **CN 2.7% vs 20.0% ↓ 0.7% share** (Chinese scholars rarely post YouTube), Europe 11.8% / LatAm 10.8% over-share — any charisma finding is Western-skewed.
- **vs other fellowships:** no external dump, so we use `region` — the US/CN bridge *is* Schwarzman (61% US+CN vs Rhodes general). Filter `region=="CN"` or `cluster==1` to see the bridge door.


In [ ]:
# One-liner to make your own plot with shorthand
import matplotlib.pyplot as plt
# join bios cluster + video warmth via shorthand/name
merged = pd.merge(pd.read_csv(DATA/"bios_clusters.csv", encoding="utf-8"), legend, left_on="name", right_on="name", how="inner", suffixes=("_bio","_vid"))
# example: PCA colored by warmth for the 24 who have both
has_both = merged[merged["warmth_v2"].notna()]
print(f"bios+video overlap: {len(has_both)} scholars have both a bio and a scored video")
if len(has_both)>0:
    # we need pca coords
    pca_df = pd.read_csv(DATA/"bios_clusters_pca.csv", encoding="utf-8")
    has_both = has_both.merge(pca_df[["name","pca0","pca1"]], on="name")
    plt.figure(figsize=(10,6))
    # Ensemble: slate to crimson encodes warmth (darker/redder = warmer), not Blues/viridis
    try:
        cmap = ss.CMAP_WARMTH
    except:
        cmap = "Greys"
    sc = plt.scatter(has_both["pca0"], has_both["pca1"], c=pd.to_numeric(has_both["warmth_v2"], errors="coerce"), cmap=cmap, s=100, edgecolor=ss.CANVAS if "ss" in globals() else "white", linewidth=0.6)
    plt.colorbar(sc, label="Warmth v2")
    for _, r in has_both.iterrows():
        plt.annotate(r["shorthand_vid"] if "shorthand_vid" in r else r["shorthand_bio"], (r["pca0"], r["pca1"]), fontsize=7, weight="bold")
    try:
        ss.style_axes(plt.gca(), title="Where Warmth Lives in the 4 Doors (Bios PCA + Video Warmth)", subtitle="Warmth v2 slate to crimson · area/position = structure, hue = warmth (Visual Hierarchy)", grid_axis="both")
    except:
        plt.title("Where warmth lives in the 4 doors (bios PCA + video warmth)")
    plt.xlabel("PCA 1"); plt.ylabel("PCA 2"); plt.tight_layout()
    # save with ensemble white canvas + tight border like Ref 2
    try:
        plt.savefig(AD/"custom_pca_warmth_shorthand.png", dpi=ss.SAVEDPI, bbox_inches="tight", facecolor=ss.CANVAS)
    except:
        plt.savefig(AD/"custom_pca_warmth_shorthand.png", dpi=300, bbox_inches="tight", facecolor="white")
    print("saved custom_pca_warmth_shorthand.png")
    display(Image(filename=str(AD/"custom_pca_warmth_shorthand.png"), width=800))


### Appendix — Shorthand legend: 74 codes, all disclosed (AA18 to XR27)

`AA18` = **A**bdullah **A**lmiqasbi, cohort 2018. Take first letter of first name + first letter of last name + cohort year. Use the code on every axis; join back to `data/video_legend.csv` on `shorthand` or `vid`.

- One film shared: `SL26` Stephanie Lin (US, Harvard, 2026) and `SL27` Stephanie Li (CN, Duke Kunshan/Columbia, 2027) both point to `imwXwnyzpRU`.
- `has_bio`: yes/— . 54 of 285 bios exist (19%). Blank warmth = 50 pending, still unscored.
- Full table below is sortable; it matches `data/video_features_all.csv` (`scored`/`pending`).


In [ ]:
# Shorthand legend — disclose all 74 (ensemble table, no rainbow)
import pandas as pd
from IPython.display import HTML, display
legend = pd.read_csv(DATA/"video_legend.csv", encoding="utf-8")
legend["yt"] = legend["youtube"].str.extract(r"(watch\?v=|shorts/)([^&]+)")[1]
# build disclosure table with full mapping
tbl = legend[["shorthand","name","cohort","country","region","university","has_bio","warmth_v2","sentiment"]].copy()
tbl = tbl.sort_values(["cohort","shorthand"])
tbl["warmth_v2"] = tbl["warmth_v2"].apply(lambda x: f"{x:.1f}" if pd.notna(x) else "— pending")
tbl["sentiment"] = tbl["sentiment"].apply(lambda x: f"{x:.2f}" if pd.notna(x) else "—")
tbl["has_bio"] = tbl["has_bio"].map({1:"yes", 0:"—"})
display(HTML(f'<div style="background:#F8FAFC; border:1px solid #E2E8F0; border-radius:8px; padding:8px 10px; font-size:11px; color:#334155; margin:10px 0;"><b style="color:#0F172A;">74 intros, 73 unique films</b> — 24 scored (Warmth v2) + 50 pending — 2018×5 · 2019×3 · 2020×6 · 2021×4 · 2022×10 · 2023×14 · 2024×5 · 2025×8 · 2026×8 · 2027×11 — <span style="color:#A51C30;">SL26/SL27 share one YouTube ID</span>.</div>'))
# cohort color bar — tiny crimson dot for scored, muted for pending (Visual Hierarchy)
def _row_style(row):
    is_pending = "pending" in str(row["warmth_v2"])
    col = "#F8FAFC" if is_pending else "white"
    return ["background:"+col+"; color:"+("#94A3B8" if is_pending else "#334155")+"; padding:4px 6px; border-bottom:1px solid #F1F5F9;"]*len(row)
styler = tbl.style.apply(_row_style, axis=1).set_table_styles([
    {"selector":"th", "props":"background:#0F172A; color:white; padding:6px 8px; text-align:left; font-size:11px; position:sticky; top:0;"},
    {"selector":"table", "props":"border-collapse:collapse; width:100%; font-family: Helvetica Neue, Inter, sans-serif; font-size:11px;"}
]).hide(axis="index")
display(HTML('<div style="max-height:520px; overflow:auto; border:1px solid #E2E8F0; border-radius:8px;">' + styler.to_html() + '</div>'))
display(HTML('<div style="font-size:10px; color:#64748B; font-style:italic; margin-top:6px;">Source: <code>data/video_legend.csv</code> → <code>data/video_features_all.csv</code> → <code>docs/VIDEO_LEGEND.md</code>. Shorthand = First+Last+cohort (First+Last+cohort). Join on <code>shorthand</code> or <code>vid</code> for any PNG axis.</div>'))
# also print compact comma list for copy-paste into code
print("shorthands (74, in cohort order):", ", ".join(legend.sort_values(["cohort","shorthand"])["shorthand"].tolist()))
print("unique vids (73):", legend["vid"].nunique(), "— duplicate:", legend[legend.duplicated("vid", keep=False)][["shorthand","vid","name","cohort"]].to_string(index=False))


## 4) What they actually accept — 5 hypotheses + how to sway your fit
See `docs/HYPOTHESES.md` for the full write-up. TL;DR:

- **H1 Four doors, not one** — Tech 36%, Policy 29%, Climate 25%, Health 9% (stable 2026 vs 2027). Don’t blend doors — hybrids blur in PCA middle.
- **H2 Bridge > brilliance** — `china/global/international` top-10 in *every* cluster. Your SOP needs one US↔China translation moment (Geneva-style).
- **H3 Verb > noun** — `founded 73, led 71` beat `passionate 58`. Show 3 verbs with numbers.
- **H4 University as signal, not filter** — Harvard loads only in Policy; Tech/Climate are feeder-dispersed (Fresno `CS27`, Montana `DM27` win there).
- **H5 Video as compensatory** — Warm Africa/LatAm over-share 10-14% vs CN 0.7%; if you’re under-represented, a 60s YouTube intro is leverage.

**To sway:** filter `data/bios_clusters_with_region.csv` where `cluster==your door` and `region==your region`, read 5 bios in that slice — copy *sentence shape*, not content. Then `python scripts/bios_nlp.py` on your draft to see which cluster you land in.


In [ ]:
# --- Validate the repo (what generate_plots.py and validate.py do) ---
import subprocess, sys
print("Running scripts/validate.py...")
try:
    out = subprocess.run([sys.executable, str(ROOT/"scripts/validate.py")], capture_output=True, text=True, timeout=30)
    print(out.stdout); print(out.stderr)
except Exception as e:
    print("validate err", e)

print("\nDocs:")
for p in [ROOT/"docs/HYPOTHESES.md", ROOT/"docs/VIDEO_LEGEND.md", ROOT/"README.md"]:
    print(p.relative_to(ROOT), p.stat().st_size)


## Next — fill the 50 pending videos (free Whisper)
```bash
# all 74, checkpointed per-video to data/meta/*.json + data/transcripts/*.txt
python scripts/video_pipeline.py            # skip existing (24)
python scripts/video_pipeline.py --force    # re-score with fear/sad penalty
make transcripts  # alias
# then re-run this notebook top-to-bottom — legend fills NA → scores, all74 plots auto-complete
```

The hybrid 4K treemap you shared (`USA/China/global/...`) and the L-system tree remain the visual essay — the 4 clusters are the *why* behind those big blocks. Use `shorthand` on every new PNG you make so reviewers can join back to `data/video_legend.csv` in one glance.
